# Further Series and Iterative Methods
#### Accompanies Section 1.5 of *Experimental Mathematics: A Computational Perspective* by Matthew P. Richey and Matthew L. Wright

## Ramanujan's Formula
Here is Ramanujan's formula, from equation (1.12) in the text:
$$ \frac{1}{\pi}=\frac{2\sqrt{2}}{9801}\sum_{k=0}^{\infty}\frac{(4k)!(1103+26390k)}{(k!)^4 396^{4k}} $$

We will write a module that adds up $m+1$ terms (indexes $k=0, ..., m$) of this sum. Since the formula gives an approximation to $1/\pi$, we must invert the result to obtain an approximation to $\pi$. The following module includes a print statement that prints each term of the sum, which is helpful for testing the code and confirming that it works as expected.

In [ ]:
import math
import mpmath

# set precision
mpmath.mp.dps = 50

def ramanujan(m):
    total = mpmath.mpf(0)
    for k in range(m + 1):
        numerator = mpmath.mpf(math.factorial(4*k) * (1103 + 26390*k))
        denominator = mpmath.mpf((math.factorial(k)**4) * (396**(4*k)))
        print(numerator, "/", denominator)
        total += numerator/denominator

    total *= mpmath.mp.sqrt(2)* 2 / 9801
    return 1 / total

Try it with just one term ($m=0$). Note that the code prints 1103 since this is the value of the first term.

In [ ]:
ramanujan(0)

Now try it with two terms. Note that the fraction that appears in the output is the value of the $k=1$ term of the sum.

In [ ]:
ramanujan(1)

It looks like our code works. Now we can remove the print statement and run the code with more terms. The following copy of the module has the print statement commented out.

In [ ]:
def ramanujan(m):
    total = mpmath.mpf(0)
    for k in range(m + 1):
        numerator = mpmath.mpf(math.factorial(4*k) * (1103 + 26390*k))
        denominator = mpmath.mpf((math.factorial(k)**4) * (396**(4*k)))
        #print(numerator, "/", denominator)
        total += numerator/denominator

    total *= mpmath.mp.sqrt(2)* 2 / 9801
    return 1 / total

Let's run the module for each value of $m$ from 0 to 10 and collect the results in a list. Using a list comprehension is convenient for this.

In [ ]:
# the following calculations requires high precision
mpmath.mp.dps = 100

piValues = [ramanujan(m) for m in range(11)]
piValues

### Accuracy

In [ ]:
import pandas as pd

accuracy = []
for val in piValues:
    diff = mpmath.fabs(val - mpmath.pi)
    if diff == 0:
        accuracy.append(float('inf'))
    else:
        accuracy.append(mpmath.floor(-1*mpmath.log10(diff)))

df = pd.DataFrame({
    'k': range(11),
    'approximation': piValues,
    'accuracy': accuracy
})
df

It appears that the Ramanujan algorithm adds about 8 digits of accuracy for each additional term of the sum.

## Salamin-Brent Algorithm
The following module implements Algorithm 1.23 in the text. Discovered independently by Richard Brent and Eugene Salamin, this iterative algorithm converges quadratically.

First we initialize the values $a_0 = 1$, $b_0 = \frac{1}{\sqrt{2}}$, and $s_0 = \frac{1}{2}$.

For $k \ge 1$, we compute the next iterations as follows:
$$ a_k = \frac{a_{k-1} + b_{k-1}}{2} $$
$$ b_k = \sqrt{a_{k-1} b_{k-1}} $$
$$ s_k = s_{k-1} - 2^k(a_k^2 - b_k^2) $$

The $k$th approximation to $\pi$ is given by evaluating:
$$ p_k = \frac{2 a_k^2}{s_k} $$

In [ ]:
def salaminBrent(m):
    a = [0] * (m + 1)
    b = [0] * (m + 1)
    s = [0] * (m + 1)
    p = [0] * (m + 1)

    a[0] = mpmath.mpf(1)
    b[0] = mpmath.mpf(1) / mpmath.mp.sqrt(2)
    s[0] = mpmath.mpf(1/2)

    # main loop
    for k in range(1, m + 1):
        a[k] = (a[k-1] + b[k-1]) / 2
        b[k] = mpmath.mp.sqrt(a[k-1] * b[k-1])
        s[k] = s[k-1] - (2**k) * (a[k]**2 - b[k]**2)
        p[k] = 2 * (a[k]**2) / s[k]

    return p[m]

Let's try calculating the first iteration of the algorithm to see if it behaves as expected:

In [ ]:
salaminBrent(1)

Let's run the algorithm again for each $m$ up to 10 and print decimal versions of the approximations we compute

In [ ]:
# the following calculations requires high precision
mpmath.mp.dps = 1200

piValues_SB = [salaminBrent(m) for m in range(1, 11)]
piValues_SB

In [ ]:
accuracy_SB = []

for val in piValues_SB:
    diff = mpmath.fabs(val - mpmath.pi)
    if diff == 0:
        accuracy.append(float('inf'))
    else:
        accuracy_SB.append(mpmath.floor(-1*mpmath.log10(diff)))

df_SB = pd.DataFrame({
    'm': range(1, 11),
    'approximation': piValues_SB,
    'accuracy': accuracy_SB
})
df_SB

In [ ]:
import matplotlib.pyplot as plt

# Plotting up to the point where standard floating point precision limits out (ignoring 'inf')
plot_acc = [a for a in accuracy_SB if a != float('inf')]
plt.figure(figsize=(6, 4))
plt.plot(range(1, len(plot_acc) + 1), plot_acc, marker='o')
plt.grid(True)
plt.show()

If the accuracy doubles with each iteration, then the plot above has a shape like $2^n$. In this case, a $\log_2$ transform of the data points should result in a linear plot. Let's plot the $\log_2$ of each data point:

In [ ]:
log2_accuracy = [math.log2(a) for a in plot_acc]

plt.figure(figsize=(6, 4))
plt.plot(range(1, len(log2_accuracy) + 1), log2_accuracy, marker='o')
plt.grid(True)
plt.show()